#Python - Non-technical introduction

##Exercise: "Play with the Pros: How to use SQL for data wrangling?"

This file provides exercises accompanying the lectures.

*Note:* If you want to make changes to this document, you need to save your own copy using the "Save copy in Drive" command in the "File" menu.

Replace  **\*\*\*Your-code-here\*\*\*** with your code.

###Preparation

**Make sure to run the follwing code before continuing.** Code will prepare everything (load data, ...).

In [1]:
#load the sqlite3 package
import sqlite3
#load the numpy package
import numpy as np
#load the Pandas package
import pandas as pd
# load wget
import wget


#Get the database file form Github and store it in the local working directory
wget.download("https://raw.githubusercontent.com/bachmannpatrick/Python-Class/master/data/database.sqlite")

#Read the csv file and store it in the variable "myData". Note: this file is hostes in a GitHub repository.
myData=pd.read_csv(filepath_or_buffer="https://raw.githubusercontent.com/bachmannpatrick/Python-Class/master/data/transactions.csv", sep=",")
#Adjust the format of column "TransDate" to datetime
myData["TransDate"]  = pd.to_datetime(myData["TransDate"], dayfirst=True)

### 27 -  Connecting to a database

1.	Find all available tables in the database database.sqlite


In [2]:
#Step 1:
#Open a connection to the database, create a cursor, and write the read_sql_query() command to show all the transactions data.
sqlite_file="database.sqlite" # Path of SQLite-Database
con=sqlite3.connect(sqlite_file) # Build a connection to the sqlite database
# Get the cursor object for the connection to traverse the relational database.
cursor=con.cursor()

In [3]:
#Step 2:
# Show all tables in the database
cursor.execute("SELECT name FROM sqlite_master;") # With the execute() method, SQL statements can be directly executed.
cursor.fetchall()

[('transactions',), ('demographics',)]

2.	Use functions of the sqlite3 module and simple SQL statements to solve the following exercises:
- List all entries of the table "demographics".


In [4]:
pd.read_sql_query("SELECT * from demographics;", con)

,Customer,Gender,Birthdate,ZIP,JoinDate
0,80365,f,1991-08-26 00:00,US-06332,2009-09-15 00:00
1,42886,f,1987-05-04 00:00,US-08055,2011-06-12 00:00
2,84374,m,1977-07-10 00:00,US-06400,1988-08-10 00:00
3,42291,m,1963-07-12 00:00,US-04533,1998-07-23 00:00
4,100001,m,1974-05-08 00:00,US-02332,1992-02-21 00:00
...,...,...,...,...,...
99779,200995,f,1992-11-09 00:00,US-62035,1978-01-11 00:00
99780,200996,m,1976-08-26 00:00,US-17844,2005-04-05 00:00
99781,200997,f,1997-06-21 00:00,US-30324,1995-10-09 00:00
99782,200998,m,1967-05-02 00:00,US-10017,1988-06-11 00:00


- Store myData in the database and name it myData_copy.

In [11]:
#myData.to_sql(con=con, name="myData_copy", index=False)
#Let's have a look at the updated database
cursor.execute("SELECT name FROM sqlite_master;")
cursor.fetchall()

[('transactions',), ('demographics',), ('myData_copy',)]

- Remove myData_copy from the database.

In [13]:
cursor.execute("DROP TABLE myData_copy")
con.commit()

In [14]:
con.close() #We are done. Let's close the database connection

###28 - Databases ant the role of SQL

No exercises for this unit.

###29 - Select operations in SQL

1.	Open a connection to the database, create a cursor for the connection and show all tables.  Then use Pandas' read_sql_query() to read in the transactions table.

In [16]:
sqlite_file="database.sqlite" # Path of SQLite-Database
con=sqlite3.connect(sqlite_file) # Build a connection to the sqlite database
# Get the cursor object for the connection to traverse the relational database.
cursor=con.cursor()
# Show all tables
cursor.execute("SELECT name FROM sqlite_master;") # With the execute() method, SQL statements can be directly executed.
cursor.fetchall()
#%% Show transactions in the "transactions" tables
pd.read_sql_query("SELECT * FROM transactions;", con)

,Customer,TransDate,Quantity,PurchAmount,Cost,TransID,TransKey
0,149332,2005-11-15 00:00,1,199.95,107.00,127998739,100000
1,172951,2008-08-29 00:00,1,199.95,108.00,128888288,100001
2,120621,2007-10-19 00:00,1,99.95,49.00,125375247,100002
3,149236,2005-11-14 00:00,1,39.95,18.95,127996226,100003
4,149236,2007-06-12 00:00,1,79.95,35.00,128670302,100004
...,...,...,...,...,...,...,...
223186,199997,2012-09-17 00:00,1,29.95,13.80,132481149,323186
223187,199997,2012-09-17 00:00,1,29.95,13.80,132481149,323187
223188,199998,2012-09-17 00:00,1,29.95,13.80,132481154,323188
223189,199999,2012-09-17 00:00,1,179.95,109.99,132481165,323189


2.	Show the IDs of all customers who purchased on 06.06.2012. How many customers purchased on that date? Please use DISTINCT Customer in your SQL query to account for duplicates.

In [ ]:
pd.read_sql_query("SELECT DISTINCT Customer FROM transactions WHERE date(TransDate) == date('2012-06-06');", con) #Using DISTINCT we get rid of the dublicates.
# 60 distinct customers

,Customer
0,198997
1,170451
2,163179
3,198991
4,150982
5,103704
6,119064
7,169320
8,149437
9,143802


3.	Calculate the new variable NetSales, which is the difference between purchase amount and cost per transaction. Check your results by taking a look at the whole table.

In [31]:
cursor.execute("ALTER TABLE transactions ADD NetSales;")# Add empty column
cursor.execute("UPDATE transactions SET NetSales=PurchAmount-Cost;") # Set values
con.commit() # Commit changes
#%% Check your results by taking a look at the whole table.
pd.read_sql_query("SELECT * from transactions;", con).head(3)

,Customer,TransDate,Quantity,PurchAmount,Cost,TransID,TransKey,NetSales
0,149332,2005-11-15 00:00,1,199.95,107.0,127998739,100000,92.95
1,172951,2008-08-29 00:00,1,199.95,108.0,128888288,100001,91.95
2,120621,2007-10-19 00:00,1,99.95,49.0,125375247,100002,50.95


In [32]:
con.close() #We are done. Let's close the database connection

###30 - Aggregate operations in SQL

0. Connect to the database again

In [33]:
sqlite_file="database.sqlite" # Path of SQLite-Database
con=sqlite3.connect(sqlite_file) # Build a connection to the sqlite database
# Get the cursor object for the connection to traverse the relational database.
cursor=con.cursor()

1.	Show for each customer the latest transaction date (LatestPurch) as well as the maximum purchase amount per transaction (MaxPurch). Please use  MAX(VariableName) here.

In [34]:
pd.read_sql_query("SELECT Customer, MAX(date(TransDate)) AS LatestPurch, MAX(PurchAmount) AS MaxPurch FROM transactions GROUP BY Customer;", con)

,Customer,LatestPurch,MaxPurch
0,100001,2011-08-24,199.95
1,100002,2004-12-29,499.95
2,100003,2012-01-23,249.95
3,100004,2012-08-05,499.95
4,100005,2010-12-19,79.95
...,...,...,...
98775,199995,2012-09-17,29.95
98776,199996,2012-09-17,179.95
98777,199997,2012-09-17,29.95
98778,199998,2012-09-17,29.95


2.	Calculate the average purchase amount (AvgPurch) and average cost (AvgCost) per year. Please use AVG(VariableName) here.

In [38]:
pd.read_sql_query("SELECT DATE (TransDate, 'start of year') AS Year, AVG(PurchAmount) AS AvgPurch, AVG(Cost) AS AvgCost from transactions GROUP BY Year;", con)

,Year,AvgPurch,AvgCost
0,2004-01-01,65.928162,31.966158
1,2005-01-01,77.364948,37.129070
2,2006-01-01,84.117115,40.217183
3,2007-01-01,87.285891,41.403788
4,2008-01-01,84.475157,39.446685
5,2009-01-01,83.087542,38.009686
6,2010-01-01,84.041538,37.891392
7,2011-01-01,84.353225,37.785226
8,2012-01-01,89.455530,40.981603


3.	Write a query for a table that shows the number of transactions (Transactions) and the average purchase amount (AvgPurch) per customer per month.

In [41]:
pd.read_sql_query("SELECT Customer, date(TransDate, 'start of month') AS MONTH, count(*) AS Transactions, AVG(PurchAmount) AS AvgPurch FROM transactions GROUP BY Customer, MONTH;", con)

,Customer,MONTH,Transactions,AvgPurch
0,100001,2011-06-01,1,79.95
1,100001,2011-08-01,1,199.95
2,100002,2004-12-01,1,499.95
3,100003,2012-01-01,2,189.95
4,100004,2012-08-01,1,499.95
...,...,...,...,...
132659,199995,2012-09-01,3,29.95
132660,199996,2012-09-01,1,179.95
132661,199997,2012-09-01,6,29.95
132662,199998,2012-09-01,1,29.95


In [42]:
con.close() #We are done. Let's close the database connection

###31 - Merge operations in SQL

0. Connect to the database again.

In [43]:
sqlite_file="database.sqlite" # Path of SQLite-Database
con=sqlite3.connect(sqlite_file) # Build a connection to the sqlite database
# Get the cursor object for the connection to traverse the relational database.
cursor=con.cursor()

1.	Join all transactions with the demographic details of the corresponding customers. What is the share of female customers in the overall purchase amount?

In [48]:
join_1 = pd.read_sql_query("SELECT Gender,(count(Gender)* 100.0 / (Select count(*) FROM transactions)) as Gender FROM transactions INNER JOIN demographics ON transactions.Customer = demographics.Customer GROUP BY Gender;", con)

2.	Compare the inner and the left outer join of the transaction and the demographic data. For how many transactions is no demographic information available?

In [ ]:
join_2=pd.read_sql_query("SELECT * FROM transactions LEFT JOIN demographics ON transactions.Customer = demographics.Customer;", con)
len(join_2)-len(join_1)
# 223188 transactions

223188

3.	Merge the transaction and the demographic data in a way, that the resulting table only shows the transactions on the day the customer joined the company.

In [50]:
pd.read_sql_query("SELECT * FROM transactions INNER JOIN demographics ON transactions.Customer = demographics.Customer AND transactions.TransDate = demographics.JoinDate;", con)

,Customer,TransDate,Quantity,PurchAmount,Cost,TransID,TransKey,NetSales,Customer,Gender,Birthdate,ZIP,JoinDate
0,170823,2008-04-04 00:00,2,109.90,45.20,128786894,100214,64.70,170823,m,1998-03-21 00:00,US-99515,2008-04-04 00:00
1,103853,2006-06-07 00:00,1,159.95,81.11,124416583,101154,78.84,103853,f,1990-08-10 00:00,US-94114,2006-06-07 00:00
2,102521,2005-09-14 00:00,1,159.95,120.00,124386341,101521,39.95,102521,m,1996-07-19 00:00,US-94803,2005-09-14 00:00
3,145017,2005-01-31 00:00,1,299.95,215.00,127858059,101804,84.95,145017,m,1987-11-14 00:00,US-29926,2005-01-31 00:00
4,145017,2005-01-31 00:00,1,0.00,0.00,127858059,101805,0.00,145017,m,1987-11-14 00:00,US-29926,2005-01-31 00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1010,199292,2012-07-29 00:00,1,24.95,13.05,131940739,321753,11.90,199292,m,1998-03-06 00:00,US-78934,2012-07-29 00:00
1011,185716,2012-02-08 00:00,1,159.95,75.13,129522431,321844,84.82,185716,f,1994-10-05 00:00,US-46725,2012-02-08 00:00
1012,199426,2012-01-09 00:00,1,39.95,20.00,131960522,322651,19.95,199426,f,1997-01-23 00:00,US-34238,2012-01-09 00:00
1013,199426,2012-01-09 00:00,1,19.95,9.50,131960522,322652,10.45,199426,f,1997-01-23 00:00,US-34238,2012-01-09 00:00


### Excursus - Connecting to Apache Spark

Run the following code to install Spark and test an exemplary Spark session.



In [51]:
# Download Java Virtual Machine (JVM)
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

The system cannot find the path specified.


In [52]:
# Install Spark
!pip install pyspark
!pip install -q findspark

^C



[notice] A new release of pip is available: 26.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
df = spark.createDataFrame([{"hello": "world"} for x in range(1000)])
df.show(3)

+-----+
|hello|
+-----+
|world|
|world|
|world|
+-----+
only showing top 3 rows

